# 2.2 — Giảm chiều dữ liệu Hotel Booking Demand

PCA phân tích phương sai. LDA dùng nhãn train để tìm một trục phân biệt hai lớp.

### Chạy thực nghiệm

Dùng bản unscaled của mục 2.1 và chỉ học bộ chuẩn hóa trên train. Nếu chạy ngoài môi trường đám mây, đặt CSV tại data/processed hoặc sửa input_path bên dưới.


In [ ]:
from pathlib import Path
import subprocess
import sys

cwd = Path.cwd()
root = cwd.parent if cwd.name == "notebooks" else cwd
input_path = root / "data/processed/clean_bookings_unscaled.csv"
if not input_path.is_file():
    uploaded = sorted((root.parent / "attachments").glob("*/drive-download-20261128T095009Z-1-001.zip"))
    if len(uploaded) == 1:
        input_path = uploaded[0]
if not input_path.is_file():
    raise FileNotFoundError("Set input_path to the unscaled CSV from section 2.1 or its ZIP.")
subprocess.run([sys.executable, str(root / "src/dim_reduction.py"), str(input_path)],
               cwd=root, check=True)

### Đọc các bảng

Chọn số trục PCA theo phương sai tích lũy. Bảng validation so sánh quan hệ của từng trục với nhãn; AUC dùng chiều đã xác định ở train.

In [ ]:
import pandas as pd
from IPython.display import Image, display

tables = root / "results/tables/02_dim_reduction"
variance = pd.read_csv(tables / "pca_variance.csv")
cutoffs = [variance["cumulative_ratio"].ge(level).idxmax() for level in (0.80, 0.90, 0.95)]
display(variance.iloc[sorted(set(range(4)).union(cutoffs))])
display(pd.read_csv(tables / "axis_validation.csv").round(4))

### Trực quan hóa

Mục này vẽ sáu cặp trong bốn trục PCA đầu và mật độ trên LD1. LDA có một chiều vì dữ liệu có hai lớp.

In [ ]:
figures = root / "results/figures/02_dim_reduction"
for name in ("pca_variance.png", "pca_pairs.png", "lda_distribution.png"):
    display(Image(filename=str(figures / name)))